# Дообучение больших языковых моделей. Квантизация и адаптеры. SFT.

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://huggingface.co/docs/transformers/en/quantization/bitsandbytes
* https://huggingface.co/docs/peft/developer_guides/quantization
* https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0
* https://huggingface.co/datasets/openai/gsm8k
* https://huggingface.co/docs/peft/main/en/quicktour
* https://discuss.pytorch.org/t/pytorch-model-size-in-mbs/149002/3
* https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftMixedModel.print_trainable_parameters
* https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftMixedModel.merge_and_unload
* https://huggingface.co/docs/trl/sft_trainer

In [38]:
# !pip install bitsandbytes datasets peft
# !pip install --quiet --upgrade transformers
# !pip install -U datasets fsspec
# !pip install trl

  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)


## Задачи для совместного разбора

1\. Обсудите понятие квантизации и рассмотрите примеры с использованием `bitsandbytes`

In [39]:
import torch
from transformers import AutoModelForCausalLM

model_name = "facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 30048.65it/s]


In [40]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,), ep

In [41]:
model.lm_head.weight

Parameter containing:
tensor([[ 0.1150, -0.1438,  0.0555,  ...,  0.2146,  0.0833,  0.0669],
        [ 0.1149, -0.1438,  0.0547,  ...,  0.2145,  0.0833,  0.0669],
        [ 0.0010, -0.0922,  0.1025,  ..., -0.0402,  0.0060, -0.1078],
        ...,
        [ 0.1152, -0.1437,  0.0547,  ...,  0.2145,  0.0833,  0.0671],
        [ 0.1151, -0.1455,  0.0546,  ...,  0.2156,  0.0837,  0.0673],
        [ 0.1156, -0.1437,  0.0577,  ...,  0.2139,  0.0833,  0.0650]],
       dtype=torch.float16, requires_grad=True)

In [42]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1653.38it/s]


In [43]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (v_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (q_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (out_proj): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
          (fc2): Linear4bit(in_features=3072, out_features=768, bias=True)
          (final_layer_nor

In [44]:
model.model.decoder.layers[0].self_attn.k_proj.weight

Parameter containing:
Parameter(Params4bit([[ 65],
            [ 26],
            [ 70],
            ...,
            [103],
            [199],
            [167]], device='cuda:0', dtype=torch.uint8))

2\. Обсудите концепцию адаптеров и способы дообучения LLM с использованием адаптеров при помощи пакетов peft и trl

In [45]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (v_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (q_proj): Linear4bit(in_features=768, out_features=768, bias=True)
            (out_proj): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
          (fc2): Linear4bit(in_features=3072, out_features=768, bias=True)
          (final_layer_nor

In [46]:
768 * 3072

# 768x8 8x3072

2359296

In [47]:
768*8 + 8*3072

30720

In [48]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["v_proj", "q_proj", "fc1"]
)

peft_model = get_peft_model(model, lora_config)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2213.31it/s]


In [49]:
peft_model

PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 768, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
          (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (layers): ModuleList(
            (0-11): 12 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear4bit(in_features=768, out_features=768, bias=True)
                (v_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(i

In [50]:
peft_model.print_trainable_parameters()

trainable params: 663,552 || all params: 125,902,848 || trainable%: 0.5270


In [51]:
from datasets import Dataset

texts = [
    "The capital of France is Paris. It is known for the Eiffel Tower.",
    "Python is a programming language. It is widely used in data science.",
    "The speed of light is approximately 299,792 km/s in vacuum.",
    "Machine learning is a subset of artificial intelligence.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The Great Wall of China was built over many centuries.",
    "Neural networks are inspired by the human brain structure.",
    "The Amazon rainforest is the largest tropical rainforest on Earth.",
]

dataset = Dataset.from_dict({"text": texts})

In [52]:
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

model = prepare_model_for_kbit_training(model)

sft_config = SFTConfig(
    num_train_epochs=3,
)

stf_trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=sft_config
)
stf_trainer.train()

Truncating train dataset: 100%|██████████| 8/8 [00:00<00:00, 3272.33 examples/s]
/opt/miniforge3/envs/ocr-research/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


TrainOutput(global_step=3, training_loss=3.5477752685546875, metrics={'train_runtime': 0.9525, 'train_samples_per_second': 25.196, 'train_steps_per_second': 3.149, 'total_flos': 234528694272.0, 'train_loss': 3.5477752685546875, 'entropy': 4.501889864603679, 'num_tokens': 348.0, 'mean_token_accuracy': 0.3364197512467702, 'epoch': 3.0})

In [53]:
stf_trainer.model

PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 768, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
          (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (layers): ModuleList(
            (0-11): 12 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear4bit(in_features=768, out_features=768, bias=True)
                (v_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(i

In [54]:
stf_trainer.model.save_pretrained("my_adapter")

In [55]:
# PeftModel.from_pretrained("my_adapter")

In [56]:
# stf_trainer.model.merge_and_unload()

## Задачи для самостоятельного решения

In [57]:
import os
import json
import random
from typing import List, Dict, Any, Tuple

import torch
import torch.nn as nn
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    PreTrainedModel,
    PreTrainedTokenizer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import SFTTrainer, SFTConfig

os.makedirs("data", exist_ok=True)

random.seed(42)
torch.manual_seed(42)

<p class="task" id="1"></p>

1\. Создайте модель `TinyLLama/TinyLlama-1.1B-Chat-v1` без квантизации. Посчитайте и выведите на экран объем памяти в мегабайтах, необходимый для хранения весов модели в формате `float32`.

Создайте квантизированную версию этой модели при помощи `bitsandbytes` (`load_in_4bit`). Выясните, насколько модель стала компактнее относительно оригинальной версии в `float32`. Опишите, что изменилось в структуре модели.


- [ ] Проверено на семинаре

In [58]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model_fp32 = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu"
)

def get_model_size_mb(model: PreTrainedModel, bytes_per_param: float) -> float:
    num_params = sum(p.numel() for p in model.parameters())
    return (num_params * bytes_per_param) / (1024 ** 2)

size_fp32_mb = get_model_size_mb(model_fp32, bytes_per_param=4.0)
print(f"Объем памяти для FP32: {size_fp32_mb:.2f} МБ")

del model_fp32
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 652.01it/s]


Объем памяти для FP32: 4196.35 МБ


In [59]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

size_4bit_mb = model_4bit.get_memory_footprint() / (1024 ** 2)

print(f"Объем памяти для 4-bit: {size_4bit_mb:.2f} МБ")
print(f"Модель стала компактнее в {size_fp32_mb / size_4bit_mb:.1f} раз")

print("\nСтруктура слоя self_attn в 4-bit модели:")
print(model_4bit.model.layers[0].self_attn)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 403.79it/s]


Объем памяти для 4-bit: 712.18 МБ
Модель стала компактнее в 5.9 раз

Структура слоя self_attn в 4-bit модели:
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
)


Стандартные слои `nn.Linear` внутри трансформера (например, `q_proj`, `v_proj`, `fc1`) заменяются на `Linear4bit` из библиотеки `bitsandbytes`. Эти слои хранят веса в специальном упакованном формате `uint8` (где в одном байте лежат два 4-битных веса).

<p class="task" id="2"></p>

2\. Создайте набор данных при помощи любой современной языковой модели или скрипта на python. В случае использования скрипта решение должно содержать этот скрипт. В случае использования LLM, нужно указать, какая LLM использовалась и с каким промптом.

Датасет должен быть содержать примеры задач на линейный поиск в небольшом массиве чисел. Размер датасета выберите сами, рекомендуемый размер - несколько сотен примеров.

Датасет может выглядеть следующим образом:

```
[
  {
    "messages": [
      {"role": "user", "content": "Find 5 in [1, 5, 3]"},
      {"role": "assistant", "content": "1. Index 0 is 1 (No). 2. Index 1 is 5 (Yes). Answer: 1"}
    ]
  },
  ...
]
```

Сохраните примеры в формате JSON на диск.  При помощи пакета `datasets` загрузите датасет с диска. Выведите на экран один пример из датасета

- [ ] Проверено на семинаре

In [60]:
def generate_linear_search_dataset(num_samples: int = 500, length_a=3, length_b=15, is_el_in_arr_alpha=0.8 ,arr_max=40) -> List[Dict[str, Any]]:
    """Генерирует датасет для задачи линейного поиска."""
    dataset =[]
    for _ in range(num_samples):
        arr_length = random.randint(length_a, length_b)
        arr =[random.randint(1, arr_max) for _ in range(arr_length)]
        
        if random.random() < is_el_in_arr_alpha:
            target = random.choice(arr)
        else:
            target = random.randint(arr_max+1, 2*arr_max)
            
        user_content = f"Find {target} in {arr}"
        
        assistant_content = ""
        found_idx = -1
        for i, val in enumerate(arr):
            if val == target:
                assistant_content += f"{i + 1}. Index {i} is {val} (Yes). "
                found_idx = i
                break
            else:
                assistant_content += f"{i + 1}. Index {i} is {val} (No). "
                
        if found_idx != -1:
            assistant_content += f"Answer: {found_idx}"
        else:
            assistant_content += "Answer: Not found"
            
        dataset.append({
            "messages":[
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content}
            ]
        })
        
    return dataset

json_path = "data/linear_search.json"
synthetic_data = generate_linear_search_dataset(600)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_data, f, indent=2)

dataset = load_dataset("json", data_files=json_path, split="train")

print(json.dumps(dataset[0], indent=2))

Generating train split: 600 examples [00:00, 26966.37 examples/s]

{
  "messages": [
    {
      "role": "user",
      "content": "Find 16 in [8, 2, 18, 16, 15, 9, 7, 35, 6, 38, 28, 3, 2]"
    },
    {
      "role": "assistant",
      "content": "1. Index 0 is 8 (No). 2. Index 1 is 2 (No). 3. Index 2 is 18 (No). 4. Index 3 is 16 (Yes). Answer: 3"
    }
  ]
}


<p class="task" id="3"></p>


3\. Возьмите один пример из вашего датасета.
Примените шаблон формата инструкций (воспользуйтесь `tokenizer.apply_chat_template` без токенизации `tokenize=False`). Выведите на экран исходную структуру и полученную строку после применения шаблона.

Найдите тег, который разделяет вопрос от ответа. Найдите его индекс.

Далее еще раз примените шаблон, но теперь с токенизацией. Создайте копию этого тензора, и замените индексы всех токенов до найденного разделителя (включительно) на -100. Декодируйтет этот тензор обратно в текст, предварительно заменив -100 на `tokenizer.pad_token_id `.

In [61]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

sample = dataset[0]["messages"]

prompt_str = tokenizer.apply_chat_template(sample, tokenize=False)

print("Исходная структура")
print(json.dumps(sample, indent=2))
print("\nСтрока после apply_chat_template")
print(prompt_str)


Исходная структура
[
  {
    "role": "user",
    "content": "Find 16 in [8, 2, 18, 16, 15, 9, 7, 35, 6, 38, 28, 3, 2]"
  },
  {
    "role": "assistant",
    "content": "1. Index 0 is 8 (No). 2. Index 1 is 2 (No). 3. Index 2 is 18 (No). 4. Index 3 is 16 (Yes). Answer: 3"
  }
]

Строка после apply_chat_template
<|user|>
Find 16 in [8, 2, 18, 16, 15, 9, 7, 35, 6, 38, 28, 3, 2]</s>
<|assistant|>
1. Index 0 is 8 (No). 2. Index 1 is 2 (No). 3. Index 2 is 18 (No). 4. Index 3 is 16 (Yes). Answer: 3</s>



In [62]:
assistant_tag = "<|assistant|>\n"
split_idx = prompt_str.find(assistant_tag)

if split_idx != -1:
    print(f"\nТег '{assistant_tag.strip()}' найден на индексе: {split_idx}")
else:
    print(f"Тег не найден!")


Тег '<|assistant|>' найден на индексе: 70


In [63]:
prompt_part = prompt_str[:split_idx + len(assistant_tag)]
encoded = tokenizer(prompt_str, return_tensors="pt")

input_ids = encoded["input_ids"][0] 
prompt_length = len(tokenizer.encode(prompt_part, add_special_tokens=False))

labels = input_ids.clone()
labels[:prompt_length] = -100

decode_ids = torch.where(labels == -100, tokenizer.pad_token_id, labels)

print(tokenizer.decode(decode_ids, skip_special_tokens=False))

</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>
1. Index 0 is 8 (No). 2. Index 1 is 2 (No). 3. Index 2 is 18 (No). 4. Index 3 is 16 (Yes). Answer: 3</s>



Я так понимаю, что `</s>` - PAD токены

<p class="task" id="4"></p>

4\. Настройте `LoraConfig` из библиотеки `peft.`

Создайте `PeftModelForCausalLM` с адаптерами для слоев `q_proj` и `v_proj`. Остальные гиперпараметры выберите самостоятельно. Примените адаптер к 4-битной версии модели. Выведите на экран структуру модели и долю обучаемых параметров.


- [ ] Проверено на семинаре

In [64]:
model_4bit = prepare_model_for_kbit_training(model_4bit)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model_4bit, lora_config)

peft_model.print_trainable_parameters()

print(peft_model.base_model.model.model.layers[0].self_attn)

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
LlamaAttention(
  (q_proj): lora.Linear4bit(
    (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (lora_dropout): ModuleDict(
      (default): Dropout(p=0.05, inplace=False)
    )
    (lora_A): ModuleDict(
      (default): Linear(in_features=2048, out_features=16, bias=False)
    )
    (lora_B): ModuleDict(
      (default): Linear(in_features=16, out_features=2048, bias=False)
    )
    (lora_embedding_A): ParameterDict()
    (lora_embedding_B): ParameterDict()
    (lora_magnitude_vector): ModuleDict()
  )
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): lora.Linear4bit(
    (base_layer): Linear4bit(in_features=2048, out_features=256, bias=False)
    (lora_dropout): ModuleDict(
      (default): Dropout(p=0.05, inplace=False)
    )
    (lora_A): ModuleDict(
      (default): Linear(in_features=2048, out_features=16, bias=False)
    )
    (lora_B): 

<p class="task" id="4"></p>

5\. Используя `STFTrainer` из пакета `trl`, обучите модель (4-битная модель с адаптерами) на сгенерированном датасете. Сохраните адаптеры на диск.


- [ ] Проверено на семинаре

In [66]:
sft_args = SFTConfig(
    output_dir="data/checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=5,
    fp16=False,
    bf16=True,
    max_length=256, 
    save_strategy="no",
    load_best_model_at_end=True,
    dataset_text_field="messages"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=sft_args,
    peft_config=lora_config,
    processing_class=tokenizer
    )

trainer.train()

adapter_path = "data/my_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Адаптеры успешно сохранены в {adapter_path}")

/opt/miniforge3/envs/ocr-research/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/opt/miniforge3/envs/ocr-research/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating train dataset: 100%|██████████| 600/600 [00:00<00:00, 126888.64 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/opt/miniforge3/envs/ocr-research/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explic

Step,Training Loss
10,1.210718
20,0.587895
30,0.398699
40,0.360741
50,0.338929
60,0.298103
70,0.326902
80,0.299489
90,0.321618
100,0.300490


Адаптеры успешно сохранены в data/my_adapter


<p class="task" id="6"></p>

6\. Загрузите базовую модель с квантованием `4bit`. Сгенерируйте пример для тестирования в формате

```
[{"role": "user", "content": "..."}]
```
Примените к нему шаблон формата инструкций. Токенизируйте полученную строку и сгенерируйте продолжение текста при помощи базовой модели (не забудьте модель перевести в режим оценки).


Загрузите версию модели (тоже с квантованием `4bit`) и добавьте к ней обученный адаптер. Проверьте работу модели на том же тестовом примере. Сравните ответ с ответом базовой модели. Прокомментируйте, изменилось ли что-то по итогу дообучения.



- [ ] Проверено на семинаре

In [77]:
from IPython.display import Markdown

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

test_messages =[
    {"role": "user", "content": "Find 7 in [2, 5, 5, 5, 6, 1, 2, 3, 8, 7, 9]"}
]

inputs = tokenizer.apply_chat_template(
    test_messages, 
    tokenize=True, 
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(peft_model.device)

print("\nОтвет БАЗОВОЙ")
peft_model.disable_adapter_layers()
with torch.no_grad():
    base_outputs = peft_model.generate(
        **inputs, 
        max_new_tokens=4096, 
        temperature=0.1, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
base_response = base_outputs[0][inputs["input_ids"].shape[1]:]
display(Markdown(tokenizer.decode(base_response, skip_special_tokens=True)))


print("\nОтвет ДООБУЧЕННОЙ")
peft_model.enable_adapter_layers()
with torch.no_grad():
    ft_outputs = peft_model.generate(
        **inputs, 
        max_new_tokens=4096, 
        temperature=0.1, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
ft_response = ft_outputs[0][inputs["input_ids"].shape[1]:]
display(Markdown(tokenizer.decode(ft_response, skip_special_tokens=True)))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 502.32it/s]
Both `max_new_tokens` (=4096) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ответ БАЗОВОЙ


Yes, here are 7 numbers from the given array:

1. 2
2. 5
3. 5
4. 6
5. 1
6. 2
7. 3
8. 8
9. 7
10. 9



Both `max_new_tokens` (=4096) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ответ ДООБУЧЕННОЙ


1. Index 0 is 2 (No). 2. Index 1 is 5 (No). 3. Index 2 is 5 (No). 4. Index 3 is 5 (No). 5. Index 4 is 6 (No). 6. Index 5 is 1 (No). 7. Index 6 is 2 (No). 8. Index 7 is 3 (No). 9. Index 8 is 8 (No). 10. Index 9 is 7 (Yes). Answer: 9